# Routing

## HTTP 路由
`Starlette` 拥有一个简单但功能强大的请求路由系统。路由表定义为路由列表，并在实例化应用程序时传递。

除了使用`Starlette`类中提供的`add_route()`函数来定义路由，`Starlette`还提供了`Router`类来定义路由。使用`Router`类定义的路由可以替代Starlette类实例来启动Starlette应用，并直接提供路由访问功能。

`Router`类定义的路由需要使用`Route`类和`Mount`类配合。以下给出一个示例。

In [ ]:
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.routing import Route

async def homepage(request):
    return PlainTextResponse("Homepage")

async def about(request):
    return PlainTextResponse("About")


routes = [
    Route("/", endpoint=homepage),
    Route("/about", endpoint=about),
]

app = Starlette(routes=routes)

该endpoint参数可以是以下之一：

- 常规函数或异步函数，它接受单个`request` 参数并返回响应。
- 实现 ASGI 接口的类，例如 `Starlette` 的`HTTPEndpoint`。

In [ ]:
from starlette.routing import Mount, Route, Router
# 从项目的其他模块中引入Home类和SubApp实例等。

async def Home(request):
    return PlainTextResponse("Home")

async def User(request):
    return PlainTextResponse("User")

SubApp = [
    Route("/", endpoint=homepage),
    Route("/about", endpoint=about),
]

async def SubHome(request):
    return PlainTextResponse("SubHome")

app = Router([
	Route('/', endpoint=Home, methods=['GET']),
	Route('/users/{username}', endpoint=User, methods=['GET']),
	Route('/users/{user_id:int}', endpoint=User, methods=['GET']),
	Mount('/mount', app=SubApp),
	Mount('/sub', app=Router([
		Route('/', endpoint=SubHome, methods=['GET', 'POST'])
	]))
])


上面的示例已经展示了`Router`类的使用。其中包括如何在路由路径中使用参数，以及定义参数的类型。具体如何在Endpoint中捕获这些定义的参数，将在后面Endpoint一节中介绍。在使用路由参数时，需要注意路由路径的定义顺序，带有参数的路由应当在同样前缀的路由组中尽量靠后放置，例如有两个路由：`\user\self`和路由`\user\{username}`，**其定义顺序就十分重要，如果带有参数的路由路径先定义，那么`\user\self`就永远不会有任何响应，因为它不能捕获到任何访问**。

这里需要牢记的是Route类用于定义一个路由处理器，Mount类用于加载一套子路由。

## 路径参数
路径可以使用 URI 模板样式来捕获路径组件。

```python
    Route('/users/{username}', user)
```

默认情况下，这将捕获到路径末尾或下一个的字符`/`。
您可以使用转换器来修改捕获的内容。可用的转换器包括：

- `str`返回一个字符串，这是默认值。
- `int`返回一个 Python 整数。
- `float`返回一个 Python 浮点数。
- `uuid`返回一个 Pythonuuid.UUID实例。
- `path`返回路径的其余部分，包括任何其他/字符。

使用转换器时，需要在前面加上冒号，如下所示：

```python
Route('/users/{user_id:int}', user)
Route('/floating-point/{number:float}', floating_point)
Route('/uploaded/{rest_of_path:path}', uploaded)
```

如果您需要一个尚未定义的其他转换器，您可以创建自己的转换器。请参阅下面的示例，了解如何创建`datetime`转换器以及如何注册它：



In [ ]:
from datetime import datetime

from starlette.convertors import Convertor, register_url_convertor


class DateTimeConvertor(Convertor):
    regex = "[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}(.[0-9]+)?"

    def convert(self, value: str) -> datetime:
        return datetime.strptime(value, "%Y-%m-%dT%H:%M:%S")

    def to_string(self, value: datetime) -> str:
        return value.strftime("%Y-%m-%dT%H:%M:%S")

register_url_convertor("datetime", DateTimeConvertor())

注册后，您将能够使用它：

```python
Route('/history/{date:datetime}', history)
```

路径参数在请求中以字典的形式提供request.path_params 。

```python
async def user(request):
    user_id = request.path_params['user_id']
    ...
```


## 处理 HTTP 方法
路由还可以指定端点处理哪些 HTTP 方法：

```python
Route('/users/{user_id:int}', user, methods=["GET", "POST"])
```

默认情况下，功能端点只会接受GET请求，除非另有说明。

## 子安装路线
在大型应用程序中，您可能会发现想要根据公共路径前缀来分解路由表的各个部分。

```python
    routes = [
        Route('/', homepage),
        Mount('/users', routes=[
            Route('/', users, methods=['GET', 'POST']),
            Route('/{username}', user),
        ])
    ]
```
这种风格允许您在项目的不同部分定义路由表的不同子集。

```python
    from myproject import users, auth

    routes = [
        Route('/', homepage),
        Mount('/users', routes=users.routes),
        Mount('/auth', routes=auth.routes),
    ]
```

您还可以使用挂载功能将子应用程序包含在 Starlette 应用程序中。例如……


```python
# This is a standalone static files server:
app = StaticFiles(directory="static")

# This is a static files server mounted within a Starlette application,
# underneath the "/static" path.
routes = [
    ...
    Mount("/static", app=StaticFiles(directory="static"), name="static")
]

app = Starlette(routes=routes)


```

## 反向 URL 查找
您通常希望能够为特定路由生成 URL，例如在需要返回重定向响应的情况下。
签名：`url_for(name, **path_params) -> URL`

```python
    routes = [
        Route("/", homepage, name="homepage")
    ]

    # We can use the following to return a URL...
    url = request.url_for("homepage")
```

URL 查找可以包含路径参数...


```python
    routes = [
        Route("/users/{username}", user, name="user_detail")
    ]

    # We can use the following to return a URL...
    url = request.url_for("user_detail", username=...)

```

如果 `Mount` 包含`name`，那么子坐标应使用 `{prefix}:{name}` 样式进行反向 URL 查找。

在没有请求实例的情况下，可以对应用程序进行反向查询，但只能返回 `URL` 路径。

```python
    routes = [
        Mount("/users", name="users", routes=[
            Route("/", user, name="user_list"),
            Route("/{username}", user, name="user_detail")
        ])
    ]

    # We can use the following to return URLs...
    url = request.url_for("users:user_list")
    url = request.url_for("users:user_detail", username=...)
```



挂载的应用程序可能包含` path=...` 参数。

```python
    routes = [
        ...
        Mount("/static", app=StaticFiles(directory="static"), name="static")
    ]

    # We can use the following to return URLs...
    url = request.url_for("static", path="/css/base.css")
```

对于没有`request`实例的情况，您可以对应用程序进行反向查找，尽管这些查找只会返回 `URL` 路径。

```python
url = app.url_path_for("user_detail", username=...)
```


### 基于主机的路由
如果您想根据标头对同一路径使用不同的路线`Host`。

请注意，匹配时端口会从Host标头中移除。例如，`Host (host='example.org:3600', ...)`即使Host标头包含或不包含除`3600 ( example.org:5600、example.org)` 之外的端口，也会被处理。因此，如果您需要在 中使用端口，可以指定端口`url_for`。

有几种方法可以将基于主机的路由连接到您的应用程序


```python
    site = Router()  # Use eg. `@site.route()` to configure this.
    api = Router()  # Use eg. `@api.route()` to configure this.
    news = Router()  # Use eg. `@news.route()` to configure this.

    routes = [
        Host('api.example.org', api, name="site_api")
    ]

    app = Starlette(routes=routes)

    app.host('www.example.org', site, name="main_site")

    news_host = Host('news.example.org', news)
    app.router.routes.append(news_host)
```

URL 查询可以像路径参数一样包含主机参数

```python
    routes = [
        Host("{subdomain}.example.org", name="sub", app=Router(routes=[
            Mount("/users", name="users", routes=[
                Route("/", user, name="user_list"),
                Route("/{username}", user, name="user_detail")
            ])
        ]))
    ]
    ...
    url = request.url_for("sub:users:user_detail", username=..., subdomain=...)
    url = request.url_for("sub:users:user_list", subdomain=...)
```

## 路线优先
按顺序将传入路径与每个路线匹配。

如果一条路线可以匹配传入路径，则应注意确保在一般情况下列出更具体的路线。

例如：

In [ ]:
# Don't do this: `/users/me` will never match incoming requests.
routes = [
    Route('/users/{username}', user),
    Route('/users/me', current_user),
]

# Do this: `/users/me` is tested first.
routes = [
    Route('/users/me', current_user),
    Route('/users/{username}', user),
]

### 使用路由器实例
如果您在底层工作，您可能希望使用普通Router 实例，而不是创建Starlette应用程序。这将为您提供一个轻量级的 ASGI 应用程序，它只提供应用程序路由，而无需将其包装在任何中间件中。

In [ ]:
app = Router(routes=[
    Route('/', homepage),
    Mount('/users', routes=[
        Route('/', users, methods=['GET', 'POST']),
        Route('/{username}', user),
    ])
])


### WebSocket 路由

当使用 `WebSocket` 端点时，您应该使用`WebSocketRoute` 而不是通常的`Route`。

路径参数和反向 URL 查找的`WebSocketRoute`工作方式与 HTTP 相同`Route`，可以在上面的 HTTP路由部分中找到。




In [ ]:
from starlette.applications import Starlette
from starlette.routing import WebSocketRoute


async def websocket_index(websocket):
    await websocket.accept()
    await websocket.send_text("Hello, websocket!")
    await websocket.close()


async def websocket_user(websocket):
    name = websocket.path_params["name"]
    await websocket.accept()
    await websocket.send_text(f"Hello, {name}")
    await websocket.close()


routes = [
    WebSocketRoute("/", endpoint=websocket_index),
    WebSocketRoute("/{name}", endpoint=websocket_user),
]

app = Starlette(routes=routes)

该endpoint参数可以是以下之一：

- 异步函数，接受单个websocket参数。
- 实现 ASGI 接口的类，例如 Starlette 的WebSocketEndpoint。